<a href="https://colab.research.google.com/github/tantaiskkh/AIB-tantai/blob/main/Datasplit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd

BASE_PATH      = '/content/drive/MyDrive/Data'
DEEPFAKE_FOLDER = os.path.join(BASE_PATH, 'deepfake_video/insightface')
REAL_FOLDER     = os.path.join(BASE_PATH, 'real_video')
OUTPUT_CSV      = os.path.join(BASE_PATH, 'Meta_Data.csv')

metadata = []

# วนอ่านไฟล์ deepfake → label 1
for filename in sorted(os.listdir(DEEPFAKE_FOLDER)):
    if filename.endswith('.mp4'):
        metadata.append({
            'file_name' : filename,
            'deepfake'  : 1,
            'path'      : f'deepfake_video/insightface/{filename}'
        })

# วนอ่านไฟล์ real → label 0
for filename in sorted(os.listdir(REAL_FOLDER)):
    if filename.endswith('.mp4'):
        metadata.append({
            'file_name' : filename,
            'deepfake'  : 0,
            'path'      : f'real_video/{filename}'
        })

df = pd.DataFrame(metadata)

print(f"✅ สร้าง Meta_Data.csv สำเร็จ!")
print(f"📊 จำนวนทั้งหมด : {len(df)} คลิป")
print(f"   - Deepfake (1): {len(df[df['deepfake']==1])} คลิป")
print(f"   - Real     (0): {len(df[df['deepfake']==0])} คลิป")
print(f"\nตัวอย่างข้อมูล:")
print(df.head(10).to_string(index=False))

df.to_csv(OUTPUT_CSV, index=False)
print(f"\n💾 บันทึกที่: {OUTPUT_CSV}")

Mounted at /content/drive
✅ สร้าง Meta_Data.csv สำเร็จ!
📊 จำนวนทั้งหมด : 1008 คลิป
   - Deepfake (1): 500 คลิป
   - Real     (0): 508 คลิป

ตัวอย่างข้อมูล:
        file_name  deepfake                                         path
 deepfake_000.mp4         1  deepfake_video/insightface/deepfake_000.mp4
 deepfake_001.mp4         1  deepfake_video/insightface/deepfake_001.mp4
 deepfake_002.mp4         1  deepfake_video/insightface/deepfake_002.mp4
 deepfake_003.mp4         1  deepfake_video/insightface/deepfake_003.mp4
 deepfake_004.mp4         1  deepfake_video/insightface/deepfake_004.mp4
 deepfake_005.mp4         1  deepfake_video/insightface/deepfake_005.mp4
 deepfake_006.mp4         1  deepfake_video/insightface/deepfake_006.mp4
 deepfake_007.mp4         1  deepfake_video/insightface/deepfake_007.mp4
 deepfake_008.mp4         1  deepfake_video/insightface/deepfake_008.mp4
deepfake_0080.mp4         1 deepfake_video/insightface/deepfake_0080.mp4

💾 บันทึกที่: /content/drive/MyDrive/Data

In [ ]:
print(df.head(10).to_string(index=False))

        file_name  deepfake                                         path
 deepfake_000.mp4         1  deepfake_video/insightface/deepfake_000.mp4
 deepfake_001.mp4         1  deepfake_video/insightface/deepfake_001.mp4
 deepfake_002.mp4         1  deepfake_video/insightface/deepfake_002.mp4
 deepfake_003.mp4         1  deepfake_video/insightface/deepfake_003.mp4
 deepfake_004.mp4         1  deepfake_video/insightface/deepfake_004.mp4
 deepfake_005.mp4         1  deepfake_video/insightface/deepfake_005.mp4
 deepfake_006.mp4         1  deepfake_video/insightface/deepfake_006.mp4
 deepfake_007.mp4         1  deepfake_video/insightface/deepfake_007.mp4
 deepfake_008.mp4         1  deepfake_video/insightface/deepfake_008.mp4
deepfake_0080.mp4         1 deepfake_video/insightface/deepfake_0080.mp4


In [ ]:
from sklearn.model_selection import train_test_split

# First split: 70% train, 30% temp
train_df, temp_df = train_test_split(
    df,
    test_size=0.3,
    stratify=df['deepfake'],
    random_state=42
)

# Second split: 10% val, 20% test (จาก 30% temp)
# val:test = 10:20 = 1:2 → test_size = 2/3
val_df, test_df = train_test_split(
    temp_df,
    test_size=2/3,             # 2/3 ของ 30% = 20% ของทั้งหมด
    stratify=temp_df['deepfake'],
    random_state=42
)

print(f'Train set     : {len(train_df)} clips ({len(train_df)/len(df)*100:.1f}%)')
print(f'Validation set: {len(val_df)} clips ({len(val_df)/len(df)*100:.1f}%)')
print(f'Test set      : {len(test_df)} clips ({len(test_df)/len(df)*100:.1f}%)')

print('\n📊 Class distribution:')
for name, subset in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    n_real = len(subset[subset['deepfake']==0])
    n_fake = len(subset[subset['deepfake']==1])
    print(f'   {name:5s}: Real={n_real}, Deepfake={n_fake}')

train_df.to_csv(os.path.join(BASE_PATH, 'train_metadata.csv'), index=False)
val_df.to_csv(os.path.join(BASE_PATH, 'val_metadata.csv'), index=False)
test_df.to_csv(os.path.join(BASE_PATH, 'test_metadata.csv'), index=False)

print(f'\n💾 บันทึก train/val/test_metadata.csv ที่ {BASE_PATH}')

Train set     : 705 clips (69.9%)
Validation set: 101 clips (10.0%)
Test set      : 202 clips (20.0%)

📊 Class distribution:
   Train: Real=355, Deepfake=350
   Val  : Real=51, Deepfake=50
   Test : Real=102, Deepfake=100

💾 บันทึก train/val/test_metadata.csv ที่ /content/drive/MyDrive/Data


In [ ]:
!pip install -q datasets

from datasets import load_dataset, ClassLabel

In [ ]:
dataset_dict = load_dataset(
    'csv',
    data_files={
        'train':      os.path.join(BASE_PATH, 'train_metadata.csv'),
        'validation': os.path.join(BASE_PATH, 'val_metadata.csv'),
        'test':       os.path.join(BASE_PATH, 'test_metadata.csv')
    }
)

print('✅ Dataset loaded!')
print(dataset_dict)

print(f'\n📋 Train features:')
print(dataset_dict['train'].features)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

✅ Dataset loaded!
DatasetDict({
    train: Dataset({
        features: ['file_name', 'deepfake', 'path'],
        num_rows: 705
    })
    validation: Dataset({
        features: ['file_name', 'deepfake', 'path'],
        num_rows: 101
    })
    test: Dataset({
        features: ['file_name', 'deepfake', 'path'],
        num_rows: 202
    })
})

📋 Train features:
{'file_name': Value('string'), 'deepfake': Value('int64'), 'path': Value('string')}


In [ ]:
class_label = ClassLabel(num_classes=2, names=['real', 'deepfake'])
dataset_dict = dataset_dict.cast_column('deepfake', class_label)

print('✅ Labels converted to ClassLabel!')
print(f'\n📋 ClassLabel feature:')
print(dataset_dict['train'].features['deepfake'])

print(f'\n🏷️  Label mapping:')
for i, name in enumerate(class_label.names):
    print(f'   {i} → {name}')

print(f'\n👀 ตัวอย่าง 3 row แรก:')
for i in range(3):
    row = dataset_dict['train'][i]
    print(f'   {i}: deepfake={row["deepfake"]} ({class_label.int2str(row["deepfake"])}), file={row["file_name"]}')

Casting the dataset:   0%|          | 0/705 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/101 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/202 [00:00<?, ? examples/s]

✅ Labels converted to ClassLabel!

📋 ClassLabel feature:
ClassLabel(names=['real', 'deepfake'])

🏷️  Label mapping:
   0 → real
   1 → deepfake

👀 ตัวอย่าง 3 row แรก:
   0: deepfake=1 (deepfake), file=deepfake_0263.mp4
   1: deepfake=1 (deepfake), file=deepfake_003.mp4
   2: deepfake=0 (real), file=113.mp4
